<a href="https://colab.research.google.com/github/NourHassan5678/Assignments/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

## 1. My lane and why

### Lane 4 — CTR / Engagement Opportunity Scoring

I'm picking Lane 4 over the other three options, for three concrete reasons:

- **The data provides a rich, immediate foundation.** The lane guide's density table shows GSC clicks and GA4 sessions are present on millions of rows, and the starter CSV ships `ctr`, `avg_position`, `engagement_rate`, and `scroll_rate` directly. This volume and density allow for robust feature engineering right out of the gate, incorporating impressions, intent types, and content freshness into the model effectively.
- **The output maps to a real, boundable action.** the end goal is highly practical: generating a ranked list of CTR or engagement review candidates. "Review this page's title/meta/snippet first" is something a reviewer can act on today, especially when paired with specific reason codes (e.g., high impressions and strong position, but weak engagement). It creates a tight, measurable loop from the model's output directly to an SEO or content strategy.

- **It demands rigorous, position-adjusted methodology.** Raw CTR cannot simply be compared across pages—a page at position 2 and a page at position 9 are not competing on a level playing field. Getting Lane 4 right means building an expected-CTR-by-tier baseline and performing residual gap analysis to see how far below expectation a page sits. The challenge of controlling for these tiers, along with filtering out low-volume noise, is exactly the kind of robust analytical reasoning I want more practice with over the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call


**Research question.** Among *visible* pages — pages with enough search exposure to matter and a position where clicks are realistically available — which ones are under-capturing clicks or engagement relative to *other pages in the same position tier*, and should be queued for a title, meta, or on-page review first?

**Unit of analysis.** One row per `content_id`, scored over the fixed 90-day window the starter dataset already provides (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`). One page = one decision.

**Exact model output.** A ranked opportunity score (0–100) per page, each carrying:
- a position-tier-relative CTR/engagement gap — not raw CTR, since raw CTR isn't comparable across tiers;
- one or more reason codes, e.g. `high_impressions_low_ctr_for_tier`, `weak_engagement_for_volume`;
- a suggested action tag: rewrite title/meta, improve intent match, improve on-page engagement, or monitor.

**Who acts, and how.** A content editor or SEO reviewer works down the ranked queue, starting with the top 20–50 candidates, and opens each page to check the actual title, meta description, and intent match before deciding whether to rewrite anything.

**Cost of a wrong call.**
- *False positive* (flagged, but the page is genuinely fine): burns a reviewer's limited time — an opportunity cost, since that slot could have gone to a real problem.
- *False negative* (a genuinely under-capturing page ranks low or never surfaces): the page keeps losing clicks it could otherwise get, silently, until the next audit.
- Because review capacity is limited (a team can realistically only work through 20–50 candidates), precision at the *top* of the list matters more than being right about every page in the dataset — which is why precision@K, not plain accuracy, is the right metric for this lane.

**Why data/ML earns its place here.** A single flat rule—such as "flag anything with a CTR under some fixed number"—completely ignores that expected CTR depends heavily on position, and to a lesser degree on user intent and content type. Median CTR is drastically different across position tiers. Therefore, a flat threshold will fail in two directions: it will under-flag bleeding top-of-page URLs (whose dropped CTR still technically clears the flat threshold), and it will over-flag overperforming deep-page URLs (whose excellent CTR for their tier still falls below the flat threshold).

What's needed instead is a relative signal—a residual gap analysis that measures how far a page sits below what is typical for its specific tier. By combining multiple correlated signals (CTR, engagement rate, scroll rate, position, and volume) rather than relying on a single cutoff, we can isolate true underperformance. A simple baseline can approximate this; a model can refine it. Either way, the result is a decision-support ranking, not a causal claim that rewriting a title will magically recover clicks.

## 3. Quick look at the data (2-3 real numbers)

In [2]:
# 1. Clone your GitHub repository into Colab
!git clone https://github.com/NourHassan5678/Assignments.git

# 2. Move your working directory inside the cloned repo
import os
os.chdir("Assignments")

# 3. Quick sanity check to make sure the 'data' folder is visible
print("Files in current directory:", os.listdir())

Cloning into 'Assignments'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 98 (delta 19), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 1.84 MiB | 14.04 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Files in current directory: ['CLAUDE.md', '.github', 'docs', 'GUIDE.md', 'README.md', '.gitignore', 'skills', 'work', 'DATA_USE.md', '.git', 'requirements.txt', 'AGENTS.md', 'scripts', 'outputs', 'notebooks', 'LICENSE', 'data', 'submission', 'SETUP.md']


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
].copy()

print(f"Total pages in starter dataset: {len(df):,}")
print(f"Visible pages (>=500 impressions, position 1-20): {len(visible):,} "
      f"({len(visible) / len(df):.1%} of total)")

# --- Metric 1: median CTR by position tier ---
ctr_by_tier = visible.groupby("position_tier")["ctr"].median().sort_values(ascending=False)
print("\nMedian CTR by position tier (visible pages only):")
print(ctr_by_tier) # As we can see in output, CTR is NOT be flat across tiers, That's exactly why a single fixed CTR threshold across all pages would be unfair.


# --- Metric 2: how many visible pages would a naive flat rule flag? ---
# This is the "low_ctr_visible_page" style rule from the starter baseline,
# applied with no regard for position tier at all.
low_ctr_flag = visible[visible["ctr"] < 0.5]
print(f"\nVisible pages a flat 'ctr < 0.5' rule would flag: {len(low_ctr_flag):,} "
      f"({len(low_ctr_flag) / len(visible):.1%} of visible pages)") # as we can see from output: If an alerting system flags 81% of the data as a "problem," it’s not an optimization tool—it's a text wall of noise.
# A human reviewer would completely ignore this report because it lacks prioritization.
#This is the ultimate proof that simple flat thresholds fail in production and why an ML/residual ranking system earns its place.

# --- Metric 3: spread within a single tier ---
# Pages within the SAME tier still show a wide CTR range, that's real headroom
# for a within-tier ranking model — the differences aren't just "position did it."
top_tier = visible["position_tier"].value_counts().idxmax()  # most common tier in this slice
tier_ctr = visible[visible["position_tier"] == top_tier]["ctr"]
print(f"\nWithin the '{top_tier}' tier: CTR ranges from {tier_ctr.min():.2f} to "
      f"{tier_ctr.max():.2f} (median {tier_ctr.median():.2f}), n={len(tier_ctr)}")


Total pages in starter dataset: 30,000
Visible pages (>=500 impressions, position 1-20): 12,023 (40.1% of total)

Median CTR by position tier (visible pages only):
position_tier
page_1      0.240
top_3       0.200
striking    0.170
page_3_5    0.155
Name: ctr, dtype: float64

Visible pages a flat 'ctr < 0.5' rule would flag: 9,759 (81.2% of visible pages)

Within the 'page_1' tier: CTR ranges from 0.00 to 5.42 (median 0.24), n=7064


## 4. Careful words: what I can and can't claim

## 4. Careful Words: What I Can and Can't Claim

This project is purely observational, not experimental. I am scoring pages based on patterns in historical search performance and engagement data, not running live A/B tests on titles or metadata. Therefore, my claims must remain strictly within these boundaries:

### 🟢 What I Can Say
* **Observed Relative Underperformance:** A page's CTR is *observed* to sit below other pages in the same position tier over a specific 90-day window.
* **Directional Priority:** The final opportunity score is a *directional* ranking—a reasonable ordering of priority given the available evidence, not an absolute statement about an individual page's absolute health.
* **Decision-Support Output:** The tool serves purely as *decision-support*. It flags *where* a reviewer should focus their attention first; it does not diagnose the exact structural fix or guarantee traffic recovery.

### 🔴 What I Will Never Say
* **Causal Proof:** I cannot claim that rewriting a title or meta description *causes* a CTR increase. Proving causality requires a controlled before-and-after experiment on live pages, which this static dataset cannot provide.
* **Definitive Diagnostics:** A low score does not automatically mean the title/meta *is* the root problem. Pages can lag for reasons entirely outside of snippet text, including misaligned search intent, seasonality, technical noise, or internal keyword cannibalization (where a sibling page absorbs the queries).
* **"Predicting Google" or Reverse-Engineering:** None of this reflects a mechanism inside Google's ranking algorithm. I am modeling observed *human user behavior* (clicks given impressions), not reverse-engineering search engine weights.